# Chapitre 6 — Structuration et transformation

## 6.4 Feature Engineering basique

---

### Objectifs

À la fin de cette leçon, vous serez capable de :
- **Créer** des variables dérivées à partir de calculs simples
- **Extraire** des composantes temporelles à partir de dates
- **Discrétiser** des variables numériques avec des bornes fixes

---

## 6.4.1 Qu'est-ce que le Feature Engineering ?

Le **feature engineering** consiste à créer de nouvelles variables (features) à partir des données existantes pour améliorer l'analyse ou les performances des modèles ML.

```
┌─────────────────────────────────────────────────────────────────────┐
│                TYPES DE FEATURE ENGINEERING                         │
├──────────────────────────┬──────────────────────────────────────────┤
│   SAFE (Module 2)        │   Calculs utilisant les données de       │
│                          │   la MÊME LIGNE uniquement                │
│                          │   Ex: marge = prix - cout                 │
├──────────────────────────┼──────────────────────────────────────────┤
│   ML PIPELINE (Module 3) │   Calculs nécessitant des statistiques   │
│                          │   Ex: qcut (quantiles), get_dummies,     │
│                          │   mean encoding                           │
└──────────────────────────┴──────────────────────────────────────────┘
```

---

## 6.4.2 Variables dérivées (calculs simples)

Les **variables dérivées** sont des calculs qui n'utilisent que les colonnes de la **même ligne**. Elles sont toujours sûres car elles ne calculent pas de statistiques sur d'autres lignes.

In [ ]:
import pandas as pd
import numpy as np

# Données de produits
df = pd.DataFrame({
    'produit_id': [1, 2, 3, 4, 5],
    'prix_vente': [100, 150, 200, 80, 300],
    'cout': [60, 90, 120, 50, 180],
    'quantite_vendue': [50, 30, 20, 100, 10]
})

print("Données initiales :")
print(df)

In [ ]:
# Variables dérivées : calculs ligne par ligne
df['marge'] = df['prix_vente'] - df['cout']
df['taux_marge_pct'] = (df['marge'] / df['prix_vente'] * 100).round(2)
df['chiffre_affaires'] = df['prix_vente'] * df['quantite_vendue']
df['profit_total'] = df['marge'] * df['quantite_vendue']

print("Avec variables dérivées :")
print(df)

In [ ]:
# Autres exemples de calculs simples
df_clients = pd.DataFrame({
    'client_id': [1, 2, 3],
    'nb_achats': [10, 5, 20],
    'montant_total': [500, 1000, 800],
    'nb_retours': [1, 0, 2]
})

# Panier moyen
df_clients['panier_moyen'] = df_clients['montant_total'] / df_clients['nb_achats']

# Taux de retour
df_clients['taux_retour_pct'] = (df_clients['nb_retours'] / df_clients['nb_achats'] * 100).round(2)

# Flag client fidèle
df_clients['est_fidele'] = (df_clients['nb_achats'] >= 10).astype(int)

print(df_clients)

---

## 6.4.3 Variables temporelles

L'extraction de composantes à partir de dates est une opération **déterministe** — elle applique la même règle à chaque ligne sans calculer de statistiques.

In [ ]:
# Données avec dates
df_temps = pd.DataFrame({
    'commande_id': [1, 2, 3, 4, 5],
    'date': pd.to_datetime(['2024-01-15 10:30', '2024-02-20 14:45', 
                           '2024-03-25 18:20', '2024-04-06 09:00', 
                           '2024-05-12 22:15'])
})

print("Données initiales :")
print(df_temps)

In [ ]:
# Extraction de composantes temporelles
df_temps['annee'] = df_temps['date'].dt.year
df_temps['mois'] = df_temps['date'].dt.month
df_temps['jour'] = df_temps['date'].dt.day
df_temps['jour_semaine'] = df_temps['date'].dt.dayofweek  # 0=lundi, 6=dimanche
df_temps['heure'] = df_temps['date'].dt.hour
df_temps['trimestre'] = df_temps['date'].dt.quarter

print("Composantes temporelles :")
print(df_temps)

In [ ]:
# Variables binaires dérivées (toujours safe)
df_temps['est_weekend'] = df_temps['jour_semaine'].isin([5, 6]).astype(int)
df_temps['est_soiree'] = (df_temps['heure'] >= 18).astype(int)
df_temps['est_matin'] = (df_temps['heure'] < 12).astype(int)

print("\nAvec variables binaires :")
print(df_temps[['date', 'jour_semaine', 'est_weekend', 'heure', 'est_soiree', 'est_matin']])

---

## 6.4.4 Binning avec bornes FIXES

Le **binning** (discrétisation) transforme une variable continue en catégories. La méthode `pd.cut()` avec des **bornes fixes** est safe car les bornes sont connues à l'avance.

In [ ]:
# Données avec âges
df_age = pd.DataFrame({
    'client_id': range(1, 11),
    'age': [22, 35, 45, 28, 67, 52, 38, 19, 73, 41]
})

# Binning avec bornes FIXES (pd.cut)
df_age['tranche_age'] = pd.cut(
    df_age['age'],
    bins=[0, 25, 35, 50, 65, 100],  # Bornes définies à l'avance
    labels=['18-25', '26-35', '36-50', '51-65', '65+']
)

print("Binning avec bornes fixes :")
print(df_age)

In [ ]:
# Autre exemple : tranches horaires
df_heures = pd.DataFrame({
    'commande_id': range(1, 8),
    'heure': [8, 11, 14, 17, 20, 23, 3]
})

# Bornes fixes pour les tranches horaires
df_heures['tranche_horaire'] = pd.cut(
    df_heures['heure'],
    bins=[-1, 6, 12, 18, 24],  # -1 pour inclure 0
    labels=['Nuit', 'Matin', 'Après-midi', 'Soir']
)

print(df_heures)

---

## ✍️ Exercice 6.4 : Feature Engineering complet

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# Données clients
np.random.seed(42)
df = pd.DataFrame({
    'client_id': range(1, 101),
    'date_naissance': pd.date_range('1960-01-01', periods=100, freq='150D'),
    'date_inscription': pd.date_range('2020-01-01', periods=100, freq='10D'),
    'nb_commandes': np.random.randint(0, 50, 100),
    'montant_total': np.random.randint(0, 5000, 100)
})

print("Données initiales :")
print(df.head())
print(f"\nNombre de colonnes : {len(df.columns)}")

In [ ]:
# 1. Calculer l'âge (variable dérivée)
df['age'] = (datetime.now() - df['date_naissance']).dt.days // 365

# 2. Créer des tranches d'âge avec bornes FIXES (pd.cut)
df['tranche_age'] = pd.cut(
    df['age'], 
    bins=[0, 30, 45, 60, 100], 
    labels=['<30', '30-45', '45-60', '60+']
)

# 3. Calculer l'ancienneté en mois
df['anciennete_mois'] = (datetime.now() - df['date_inscription']).dt.days // 30

# 4. Calculer le panier moyen (attention à la division par 0)
df['panier_moyen'] = np.where(
    df['nb_commandes'] > 0, 
    df['montant_total'] / df['nb_commandes'], 
    0
)

# 5. Créer un flag "client actif" (au moins 5 commandes)
df['est_actif'] = (df['nb_commandes'] >= 5).astype(int)

print("Après feature engineering :")
print(df[['client_id', 'age', 'tranche_age', 'anciennete_mois', 
          'panier_moyen', 'est_actif']].head(10))

In [ ]:
# Vérification des nouvelles colonnes
print(f"\nNombre de colonnes final : {len(df.columns)}")
print(f"Colonnes : {list(df.columns)}")

---

## 📝 Résumé

| Type | Exemple | Safe en Module 2 ? |
|------|---------|--------------------|
| Variables dérivées | `marge = prix - cout` | ✅ OUI |
| Extraction temporelle | `df['mois'] = df['date'].dt.month` | ✅ OUI |
| Binning avec bornes fixes | `pd.cut(age, bins=[0,25,50,100])` | ✅ OUI |
| Flags binaires | `df['est_actif'] = (df['nb'] >= 5)` | ✅ OUI |

---

## ➡️ Et l'encoding et le binning par quantiles ?

Certaines transformations doivent être faites dans le **Module 3 (Pipeline ML)** car elles calculent des statistiques sur les données :

```
┌─────────────────────────────────────────────────────────────────────┐
│  Module 2 (ici)                │    Module 3 (Pipeline ML)         │
├────────────────────────────────┼───────────────────────────────────┤
│  • pd.cut() avec bornes fixes  │    • pd.qcut() → KBinsDiscretizer │
│  • Calculs ligne par ligne     │    • pd.get_dummies → OneHotEncoder│
│  • Extraction temporelle       │    • Encoding catégoriel          │
│  • Flags binaires              │    • Mean/Target encoding         │
└────────────────────────────────┴───────────────────────────────────┘
```

> 💡 **Pourquoi ?** `pd.qcut()` calcule les quantiles sur les données. Si vous le faites avant le split, vous utilisez les données de test pour définir les bornes → **data leakage**.